In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az

from sklearn.model_selection import train_test_split

from src.preprocessing.preprocessing import load_data, clean_data, prepare_features, build_preprocessing_pipeline

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [2]:
df = clean_data(load_data("../data/raw/amazon_sales_dataset.csv"))

X, y = prepare_features(df, target="discounted_price")

y_log = np.log(y.clip(lower=1e-9))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((40000, 10), (10000, 10))

In [3]:
pre = build_preprocessing_pipeline(X_train)

X_train_mat = pre.fit_transform(X_train)
X_test_mat = pre.transform(X_test)

# Convertir a denso si es sparse
if hasattr(X_train_mat, "toarray"):
    X_train_mat = X_train_mat.toarray()
    X_test_mat = X_test_mat.toarray()

X_train_mat = X_train_mat.astype("float32")
X_test_mat = X_test_mat.astype("float32")

X_train_mat.shape, X_test_mat.shape

((40000, 20), (10000, 20))

In [4]:
rng = np.random.RandomState(42)

n = X_train_mat.shape[0]
subset_size = min(1000, n)  # puedes bajar a 1000 si tarda mucho

idx = rng.choice(n, size=subset_size, replace=False)

X_train_small = X_train_mat[idx]
y_train_small = y_train.values[idx].astype("float32")

X_train_small.shape, X_train_mat.shape

((1000, 20), (40000, 20))

In [6]:
coords = {"features": np.arange(X_train_small.shape[1])}

with pm.Model(coords=coords) as model:
    X_shared = pm.Data("X", X_train_small)
    y_shared = pm.Data("y", y_train_small)

    sigma = pm.Exponential("sigma", 1.0)
    beta = pm.Normal("beta", 0.0, 0.5, dims="features")
    intercept = pm.Normal("intercept", 0.0, 2.0)

    mu = intercept + pm.math.dot(X_shared, beta)

    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_shared)

    idata = pm.sample(
        draws=400,
        tune=400,
        chains=2,
        cores=1,
        target_accept=0.9,
        random_seed=42
    )

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [sigma, beta, intercept]


Output()

Sampling 2 chains for 400 tune and 400 draw iterations (800 + 800 draws total) took 26725 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [ ]:
az.summary(idata, var_names=["intercept", "sigma"])
plt.show()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
intercept,4.911,0.388,4.226,5.694,0.026,0.017,227.0,200.0,1.03
sigma,0.375,0.009,0.359,0.392,0.000,0.000,1230.0,598.0,1.00
